# Results: proof for every claim

This notebook **proves** each result in `results_and_conclusions.md`. Every result follows the same four steps:

1. **Claim:** what we say is true.
2. **How we test it:** the test in plain words.
3. **Proof:** the code, the numbers and a chart.
4. **Verdict:** an automatic ✅ PASS or ❌ FAIL check against a clear rule.

At the end, a **scorecard** lists every check in one table.

**How to read the tests (quick guide)**

| Term | Meaning in simple words |
|---|---|
| **p-value** | The chance of seeing this result *by pure luck*. **p < 0.05 means "not luck".** |
| **95% confidence interval (CI)** | The range where the true value most likely lies. If it does not include 0 (or 1 for odds), the effect is real. |
| **Odds ratio (OR)** | How many times more likely. OR = 3 means 3× the odds. |
| **Cramér's V** | Strength of a link between two categories: 0.1 is small, 0.3 medium, 0.5 large. |
| **AUC** | How well a model separates two groups: 0.5 is guessing, 1.0 is perfect. |
| **Cross-validation** | We test the model on data it has **never seen**, five times over, so the score is honest. |
| **Bootstrap** | We re-sample the data 1,000 times to check the result is stable. |
| **Permutation test** | We shuffle the answers randomly and retrain. If the real model beats every shuffled version, it learned something real. |

*Input: the cleaned files from `data_exploration.ipynb` (`cleaned/*.csv`).*

## Setup

In [ ]:
import warnings, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FuncFormatter, LogLocator
from scipy import stats
import statsmodels.formula.api as smf
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict, permutation_test_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, make_scorer, adjusted_rand_score
from sklearn.cluster import KMeans
warnings.filterwarnings('ignore')
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 50)
RNG = 42

BLUE, ORANGE, AQUA = '#2a78d6', '#eb6834', '#1baf7a'
INK, MUTED, GRID, LIGHT = '#0b0b0b', '#52514e', '#e1e0d9', '#c3c2b7'
plt.rcParams.update({
    'figure.facecolor': '#fcfcfb', 'axes.facecolor': '#fcfcfb', 'axes.edgecolor': LIGHT,
    'axes.labelcolor': MUTED, 'axes.titlecolor': INK, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.titlelocation': 'left', 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.8, 'axes.axisbelow': True,
    'xtick.color': MUTED, 'ytick.color': MUTED, 'font.size': 10, 'legend.frameon': False, 'figure.dpi': 110})
SEQ = LinearSegmentedColormap.from_list('seq', ['#f4f8fd', '#9ec5f4', '#2a78d6', '#0d366b'])
DIV = LinearSegmentedColormap.from_list('div', ['#2a78d6', '#f0efec', '#eb6834'])

# ---- scorecard: every verdict is recorded here ----
CHECKS = []
def verdict(result, test, value, rule, passed):
    CHECKS.append({'result': result, 'test': test, 'value': value, 'pass rule': rule, 'verdict': '✅ PASS' if passed else '❌ FAIL'})
    print(f"{'✅ PASS' if passed else '❌ FAIL'}  |  {result}  |  {test}: {value}  (rule: {rule})")

def log_axis(ax):
    ax.set_xscale('log')
    ax.xaxis.set_major_locator(LogLocator(subs=(1, 2, 5)))
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:g}'))
    ax.xaxis.set_minor_formatter(FuncFormatter(lambda v, _: ''))

def forest(t, title, xlabel):
    t = t.sort_values('odds_ratio'); y = np.arange(len(t)); sig = t['p_value'] < 0.05
    fig, ax = plt.subplots(figsize=(8.5, 0.5 * len(t) + 1.3))
    ax.hlines(y, t['ci_low'], t['ci_high'], color=MUTED, lw=2)
    ax.scatter(t['odds_ratio'], y, s=75, color=np.where(sig, BLUE, LIGHT), zorder=3, edgecolor='#fcfcfb', linewidth=1.5)
    ax.axvline(1, color=INK, lw=1, ls='--'); log_axis(ax)
    ax.set_yticks(y, [i.replace('_', ' ') for i in t.index]); ax.grid(axis='y', visible=False)
    for yi in range(len(t)):
        ax.text(t['ci_high'].iloc[yi] * 1.08, yi, f"OR {t['odds_ratio'].iloc[yi]:.2f}  p={t['p_value'].iloc[yi]:.3f}", va='center', fontsize=8, color=MUTED)
    ax.set_title(title); ax.set_xlabel(xlabel + '  ·  dashed line = no effect  ·  blue = significant')
    return ax

def cramers_v(x, y):
    ct = pd.crosstab(np.asarray(x), np.asarray(y)); chi2, p, _, _ = stats.chi2_contingency(ct)
    return np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1))), p

a   = pd.read_csv('cleaned/analytics_jobs_clean.csv')
d   = pd.read_csv('cleaned/ds_jobs_clean.csv')
jds = pd.read_csv('cleaned/jds_clean.csv')
sds = pd.read_csv('cleaned/sds_clean.csv')
BANDS = ['0to3', '3to6', '6to10', '10to15', '15to25', '25to50']
a = a[~a['exp_band_conflict']].copy()
a['band_idx'] = a['salary'].map({b: i for i, b in enumerate(BANDS)})
a['high_pay'] = (a['band_idx'] >= 4).astype(int)
d['log_salary'] = np.log(d['avg_salary_L'])
top10 = d.groupby('company_name')['num_of_jobs'].sum().nlargest(10).index
d['big_hirer'] = d['company_name'].isin(top10).astype(int)
SKILLS = ['big_data_skills', 'maths_stats_skills', 'coding_skills', 'ai_and_ml_skills', 'dashboard_and_storytelling_skills']
TRAITS = ['neuroticism', 'extraversion', 'openness_to_experience', 'agreeableness', 'conscientiousness']
print('rows:', {'analytics': len(a), 'ds': len(d), 'jds': len(jds), 'sds': len(sds)})

# Stage 1: Getting hired (what decides pay?)

## Result 1: Experience is the biggest driver of pay
**Claim:** each extra year of experience adds about **12%** to salary, and experience has the **strongest link** with salary band.

**How we test it:**
1. **Regression** on log-salary, holding job title fixed, to get the % gain per year.
2. **Bootstrap:** repeat the regression on 1,000 re-sampled datasets to check the 12% is stable.
3. **Spearman correlation** between experience and salary *inside each job title*.
4. **Cramér's V** between experience and salary band in the 14.8k job postings.

In [ ]:
ols = smf.ols("log_salary ~ min_experience + C(job_title) + big_hirer", data=d).fit(cov_type='HC3')
gain = (np.exp(ols.params['min_experience']) - 1) * 100
lo, hi = (np.exp(ols.conf_int().loc['min_experience']) - 1) * 100
print(f'Regression: each year adds {gain:.1f}% (95% CI {lo:.1f}% to {hi:.1f}%), p = {ols.pvalues["min_experience"]:.1e}')

rng = np.random.default_rng(RNG); boot = []
for _ in range(1000):
    s = d.sample(len(d), replace=True, random_state=rng.integers(1e9))
    b = smf.ols("log_salary ~ min_experience + C(job_title) + big_hirer", data=s).fit().params['min_experience']
    boot.append((np.exp(b) - 1) * 100)
boot = np.array(boot)
print(f'Bootstrap (1,000 runs): middle 95% of results = {np.percentile(boot, 2.5):.1f}% to {np.percentile(boot, 97.5):.1f}%')

within = d.groupby('job_title').apply(lambda g: stats.spearmanr(g['min_experience'], g['avg_salary_L'])[0])
print(f'Spearman inside each title: min {within.min():.2f}, max {within.max():.2f} (positive in {int((within > 0).sum())} of {len(within)} titles)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
med = d[d.min_experience <= 10].groupby('min_experience')['avg_salary_L'].median()
axes[0].scatter(d['min_experience'].clip(upper=12), d['avg_salary_L'], s=10, color=BLUE, alpha=0.18)
axes[0].plot(med.index, med.values, color=ORANGE, lw=2.5, marker='o', ms=7, label='median salary')
axes[0].set_xlabel('minimum experience (years)'); axes[0].set_ylabel('avg salary (₹ lakhs)'); axes[0].set_ylim(0, 60)
axes[0].set_title('Salary climbs with every year of experience'); axes[0].legend(loc='upper left')
axes[1].hist(boot, bins=40, color=BLUE, edgecolor='#fcfcfb')
axes[1].axvline(gain, color=ORANGE, lw=2.5, label=f'our estimate {gain:.1f}%')
axes[1].axvline(0, color=INK, lw=1, ls='--', label='0% = no effect')
axes[1].set_xlabel('% salary gain per year (1,000 bootstrap runs)'); axes[1].set_title('The 12% is stable: no run comes near 0%')
axes[1].legend(loc='upper left')
plt.tight_layout(); plt.show()

v = {}
a['exp_bucket'] = pd.cut(a['exp_min'], [-1, 1, 3, 5, 8, 30])
for col in ['exp_bucket', 'role_group', 'primary_city']:
    top = a[col].value_counts().head(9).index if col == 'primary_city' else a[col].unique()
    v[col] = cramers_v(a[col].where(a[col].isin(top), 'Other').astype(str), a['salary'])[0]
print("Cramér's V with salary band:", {k: round(x, 3) for k, x in v.items()})

verdict('R1 Experience', 'gain per year (regression)', f'{gain:.1f}%  CI {lo:.1f}–{hi:.1f}', 'CI above 0', lo > 0)
verdict('R1 Experience', 'bootstrap lowest run', f'{boot.min():.1f}%', 'all runs > 0', boot.min() > 0)
verdict('R1 Experience', 'positive inside every title', f'{int((within > 0).sum())}/{len(within)}', 'all titles', (within > 0).all())
verdict('R1 Experience', "strongest Cramér's V", f"{v['exp_bucket']:.2f} vs next {max(v['role_group'], v['primary_city']):.2f}", 'experience highest', v['exp_bucket'] == max(v.values()))

**In simple words:** across 1,000 re-sampled versions of the data, the gain per year was always about 10–13% and never close to zero. Experience also beats role and city by a wide margin. **The claim holds.**

## Result 2: Job title matters, even at the same experience
**Claim:** at the same experience, a Data Scientist earns about **2.3×** (+134%) a Data Analyst, and the model explains **58%** of salary differences.

**How we test it:**
1. **Kruskal-Wallis test:** are salaries really different across the 10 titles?
2. **Regression** with Data Analyst as the baseline gives the % premium per title, with 95% CI.
3. **Cross-validated R²:** does the model still explain salary on companies it has **not** seen?

In [ ]:
H, p_kw = stats.kruskal(*[g['avg_salary_L'] for _, g in d.groupby('job_title')])
print(f'Kruskal-Wallis: H = {H:.0f}, p = {p_kw:.1e}')

ols2 = smf.ols("log_salary ~ min_experience + C(job_title, Treatment('Data Analyst')) + big_hirer", data=d).fit(cov_type='HC3')
ci = ols2.conf_int()
prem = pd.DataFrame({'premium': ols2.params, 'lo': ci[0], 'hi': ci[1]}).filter(like='job_title', axis=0)
prem = (np.exp(prem) - 1) * 100
prem.index = [re.search(r'\[T\.(.*)\]', i).group(1) for i in prem.index]
prem = prem.sort_values('premium')

# honest R²: 5-fold cross-validation, predicting companies the model never saw
from sklearn.model_selection import KFold
r2 = []
for tr, te in KFold(5, shuffle=True, random_state=RNG).split(d):
    m = smf.ols("log_salary ~ min_experience + C(job_title) + big_hirer", data=d.iloc[tr]).fit()
    pred = m.predict(d.iloc[te]); y_ = d.iloc[te]['log_salary']
    r2.append(1 - ((y_ - pred) ** 2).sum() / ((y_ - y_.mean()) ** 2).sum())
print(f'R² on training data = {ols2.rsquared:.3f} | cross-validated R² = {np.mean(r2):.3f} (folds: {np.round(r2, 2)})')

fig, ax = plt.subplots(figsize=(8.5, 4.3)); y = np.arange(len(prem))
ax.hlines(y, prem['lo'], prem['hi'], color=MUTED, lw=2)
ax.scatter(prem['premium'], y, s=75, color=BLUE, zorder=3, edgecolor='#fcfcfb', linewidth=1.5)
ax.axvline(0, color=INK, lw=1, ls='--'); ax.set_yticks(y, prem.index); ax.grid(axis='y', visible=False)
for yi, v_ in enumerate(prem['premium']):
    ax.text(prem['hi'].iloc[yi] + 5, yi, f'+{v_:.0f}%', va='center', fontsize=8, color=MUTED)
ax.set_xlabel('% more than a Data Analyst with the same experience (dot = estimate, line = 95% CI)')
ax.set_title('Every title earns more than Data Analyst; no CI touches 0')
plt.show()

ds_p = prem.loc['Data Scientist']
verdict('R2 Job title', 'Kruskal-Wallis p', f'{p_kw:.1e}', 'p < 0.05', p_kw < 0.05)
verdict('R2 Job title', 'Data Scientist premium', f"+{ds_p['premium']:.0f}%  CI {ds_p['lo']:.0f}–{ds_p['hi']:.0f}", 'CI above 0', ds_p['lo'] > 0)
verdict('R2 Job title', 'cross-validated R²', f'{np.mean(r2):.2f}', '≥ 0.50 and close to training R²', np.mean(r2) >= 0.5 and abs(np.mean(r2) - ols2.rsquared) < 0.05)

**In simple words:** every confidence line sits far to the right of zero, so every title truly pays more than Data Analyst. The model explains salary just as well on companies it never saw (cross-validated R² close to 0.58), so it is not over-fitted. **The claim holds.**

## Result 3: Specific technical skills pay more; generic keywords do not
**Claim:** 42 of 116 common skills have a real link with pay. Specific skills give **2–3×** better odds of a ₹15 L+ job, and generic keywords are linked to low pay.

**How we test it:** for each skill we compare postings **with** vs **without** it, using a chi-square test and an odds ratio.
Because we test 116 skills at once, we use the **Bonferroni correction** (multiply p by 116), so lucky hits don't count.

In [ ]:
sk = (a['key_skills'].fillna('').str.replace('...', '', regex=False).str.lower().str.split(',')
        .apply(lambda L: {s.strip() for s in L if s.strip()}))
vocab = sk.explode().value_counts(); vocab = vocab[vocab >= 100].index
rows = []
for s in vocab:
    has = sk.apply(lambda L, s=s: s in L)
    ct = pd.crosstab(has, a['high_pay']).reindex(index=[False, True], columns=[0, 1], fill_value=0) + 0.5
    or_ = (ct.loc[True, 1] * ct.loc[False, 0]) / (ct.loc[True, 0] * ct.loc[False, 1])
    se = np.sqrt((1 / ct.values).sum())
    rows.append({'skill': s, 'postings': int(has.sum()), 'odds_ratio': or_,
                 'ci_low': np.exp(np.log(or_) - 1.96 * se), 'ci_high': np.exp(np.log(or_) + 1.96 * se),
                 'p_bonf': min(stats.chi2_contingency(ct)[1] * len(vocab), 1)})
sp = pd.DataFrame(rows).set_index('skill')
n_sig = int((sp.p_bonf < 0.05).sum())
print(f'skills tested: {len(sp)} | significant after Bonferroni: {n_sig}')

show = pd.concat([sp.nlargest(8, 'odds_ratio'), sp.nsmallest(8, 'odds_ratio')]).sort_values('odds_ratio')
fig, ax = plt.subplots(figsize=(9, 6)); y = np.arange(len(show))
colors = np.where(show['odds_ratio'] > 1, BLUE, ORANGE)
ax.hlines(y, show['ci_low'], show['ci_high'], color=MUTED, lw=2)
ax.scatter(show['odds_ratio'], y, s=70, color=colors, zorder=3, edgecolor='#fcfcfb', linewidth=1.5)
ax.axvline(1, color=INK, lw=1, ls='--'); log_axis(ax)
ax.set_yticks(y, show.index); ax.grid(axis='y', visible=False)
ax.set_xlabel('odds of a ₹15 L+ job when the skill is listed (× vs without it, 95% CI)')
ax.set_title('Top 8 premium skills (blue) vs top 8 penalty skills (orange)')
plt.show()
top8, bot8 = sp.nlargest(8, 'odds_ratio'), sp.nsmallest(8, 'odds_ratio')
verdict('R3 Skills', 'significant skills (Bonferroni)', f'{n_sig} of {len(sp)}', 'many > 0', n_sig > 0)
verdict('R3 Skills', 'top-8 premium odds ratios', f"{top8.odds_ratio.min():.1f}–{top8.odds_ratio.max():.1f}×", 'all ≥ 2 and significant', (top8.odds_ratio >= 2).all() and (top8.p_bonf < 0.05).all())
verdict('R3 Skills', 'top-8 penalty odds ratios', f"{bot8.odds_ratio.min():.2f}–{bot8.odds_ratio.max():.2f}×", 'all < 0.5 and significant', (bot8.odds_ratio < 0.5).all() and (bot8.p_bonf < 0.05).all())

**In simple words:** none of the blue lines crosses the dashed "no effect" line, and neither do the orange ones. The skill gaps are real even after the strict Bonferroni correction. **The claim holds.**

## Result 4: Most jobs come from companies that pay less
**Claim:** the "high volume, low pay" segment provides about **67%** of openings, and the top-10 hirers pay about **11% less** for the same role and experience.

**How we test it:**
1. Segment companies (with ≥3 titles) by openings × median pay, then count each segment's share of openings.
2. **Regression coefficient** for `big_hirer`, which compares the same title and experience.
3. **Mann-Whitney test:** do the top-10 hirers' salaries differ from everyone else's *within the same title*?

In [ ]:
comp = d.groupby('company_name').agg(openings=('num_of_jobs', 'sum'), med=('avg_salary_L', 'median'), titles=('job_title', 'nunique'))
comp = comp[comp.titles >= 3]
mx, my = comp.openings.median(), comp.med.median()
comp['segment'] = np.select([(comp.openings >= mx) & (comp.med < my), (comp.openings >= mx) & (comp.med >= my), (comp.openings < mx) & (comp.med >= my)],
                            ['High volume, low pay', 'High volume, high pay', 'Selective, high pay'], 'Selective, low pay')
seg = comp.groupby('segment').agg(share=('openings', 'sum'), median_pay=('med', 'median'))
seg['share'] = seg['share'] / seg['share'].sum() * 100
print(seg.round(1))

bh = (np.exp(ols.params['big_hirer']) - 1) * 100
bh_lo, bh_hi = (np.exp(ols.conf_int().loc['big_hirer']) - 1) * 100
print(f'\nTop-10 hirer effect (same title + experience): {bh:.1f}%  (95% CI {bh_lo:.1f}% to {bh_hi:.1f}%)')

# within-title comparison: salary relative to the title median
d['rel_to_title'] = d['avg_salary_L'] / d.groupby('job_title')['avg_salary_L'].transform('median')
U, p_mw = stats.mannwhitneyu(d.loc[d.big_hirer == 1, 'rel_to_title'], d.loc[d.big_hirer == 0, 'rel_to_title'])
print(f'Mann-Whitney on salary ÷ title median: top-10 median {d.loc[d.big_hirer==1,"rel_to_title"].median():.2f} vs others {d.loc[d.big_hirer==0,"rel_to_title"].median():.2f}, p = {p_mw:.1e}')

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
s = seg.sort_values('share')
axes[0].barh(s.index, s['share'], color=[ORANGE if i == 'High volume, low pay' else BLUE for i in s.index], height=0.6)
for i, v_ in enumerate(s['share']): axes[0].text(v_, i, f' {v_:.0f}%', va='center', fontsize=9, color=MUTED)
axes[0].set_xlabel('% of all openings'); axes[0].set_title('Two-thirds of openings are high-volume, low-pay'); axes[0].grid(axis='y', visible=False)
bp = axes[1].boxplot([d.loc[d.big_hirer == 1, 'rel_to_title'], d.loc[d.big_hirer == 0, 'rel_to_title']], vert=False, widths=0.5,
                     patch_artist=True, showfliers=False, medianprops={'color': INK, 'lw': 2})
for patch, c in zip(bp['boxes'], [ORANGE, BLUE]): patch.set_facecolor(c); patch.set_alpha(0.75)
axes[1].set_yticks([1, 2], ['Top-10 hirers', 'All other companies']); axes[1].axvline(1, color=INK, ls='--', lw=1)
axes[1].set_xlabel('salary ÷ median salary of the same title (1 = typical)'); axes[1].set_title('Top-10 hirers pay below the title median')
axes[1].grid(axis='y', visible=False)
plt.tight_layout(); plt.show()

verdict('R4 Company', 'openings share: volume low-pay', f"{seg.loc['High volume, low pay','share']:.0f}%", 'largest segment', seg['share'].idxmax() == 'High volume, low pay')
verdict('R4 Company', 'top-10 hirer pay effect', f'{bh:.1f}%  CI {bh_lo:.1f} to {bh_hi:.1f}', 'CI below 0', bh_hi < 0)
verdict('R4 Company', 'Mann-Whitney within title', f'p = {p_mw:.1e}', 'p < 0.05', p_mw < 0.05)

**In simple words:** the biggest hirers' salaries sit below the typical salary for the same title, and the regression gap (about −11%) has a confidence interval entirely below zero. **The claim holds.**

## Result 5: City matters much less than experience
**Claim:** city has a **small** link with pay (V ≈ 0.09), far weaker than experience (V ≈ 0.40).

**How we test it:** we compare Cramér's V for experience, role and city. We also check that the gap is stable with a **bootstrap** (500 re-samples).

In [ ]:
city9 = a['primary_city'].where(a['primary_city'].isin(a['primary_city'].value_counts().head(9).index), 'Other')
rng = np.random.default_rng(RNG); bv = {'experience': [], 'city': [], 'role': []}
for _ in range(500):
    idx = rng.integers(0, len(a), len(a))
    s = a.iloc[idx]
    bv['experience'].append(cramers_v(s['exp_bucket'].astype(str), s['salary'])[0])
    bv['city'].append(cramers_v(city9.iloc[idx], s['salary'])[0])
    bv['role'].append(cramers_v(s['role_group'], s['salary'])[0])
bv = pd.DataFrame(bv)
summary = pd.DataFrame({'V': bv.mean(), 'low_2.5%': bv.quantile(.025), 'high_97.5%': bv.quantile(.975)}).round(3)
print(summary)
p_city = cramers_v(city9, a['salary'])[1]
print(f'city chi-square p = {p_city:.1e} (real, but small)')

fig, ax = plt.subplots(figsize=(8, 3)); s = summary.sort_values('V'); y = np.arange(len(s))
ax.barh(y, s['V'], color=[BLUE if i == 'experience' else LIGHT for i in s.index], height=0.55)
ax.errorbar(s['V'], y, xerr=[s['V'] - s['low_2.5%'], s['high_97.5%'] - s['V']], fmt='none', ecolor=INK, capsize=4)
for yi, v_ in enumerate(s['V']): ax.text(s['high_97.5%'].iloc[yi] + 0.01, yi, f'{v_:.2f}', va='center', fontsize=9, color=MUTED)
ax.set_yticks(y, s.index); ax.grid(axis='y', visible=False)
ax.set_xlabel("Cramér's V with salary band (bar = mean, line = 95% bootstrap range)")
ax.set_title('Experience matters about 4× more than city')
plt.show()
verdict('R5 City', 'V experience ÷ V city', f"{summary.loc['experience','V'] / summary.loc['city','V']:.1f}×", '> 3× and bootstrap ranges do not overlap',
        summary.loc['experience','V'] / summary.loc['city','V'] > 3 and summary.loc['experience', 'low_2.5%'] > summary.loc['city', 'high_97.5%'])
verdict('R5 City', "city V (effect size)", f"{summary.loc['city','V']:.2f}", '≤ 0.1 = small', round(summary.loc['city', 'V'], 2) <= 0.1)

**In simple words:** across 500 re-samples, the experience bar stays far above the city bar. City is a real effect but a small one. **The claim holds.**

## Result 6: Salary band can be predicted reasonably well
**Claim:** the model predicts the salary band **within one band about 83%** of the time, against 62% for guessing.

**How we test it:** 5-fold cross-validation. The model and a "guess the most common band" baseline are scored on the **same unseen folds**, and we check the model wins **in every fold**.

In [ ]:
top_sk = sk.explode().value_counts().head(150).index
S = pd.DataFrame({f'skill_{s}': sk.apply(lambda L, s=s: int(s in L)) for s in top_sk}, index=a.index)
city8 = a['primary_city'].where(a['primary_city'].isin(a['primary_city'].value_counts().head(8).index), 'Other')
X = pd.concat([a[['exp_min', 'exp_max', 'n_cities']].astype(float), a[['is_fresher', 'junk_posting']].astype(int),
               pd.get_dummies(city8, prefix='city', dtype=int), pd.get_dummies(a['role_group'], prefix='role', dtype=int), S], axis=1)
y = a['band_idx'].values
within_one = make_scorer(lambda t, p: np.mean(np.abs(t - p) <= 1))
cv = StratifiedKFold(5, shuffle=True, random_state=RNG)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=0.5))
m = cross_validate(model, X, y, cv=cv, scoring={'exact': 'accuracy', 'within1': within_one})
b = cross_validate(DummyClassifier(strategy='most_frequent'), X, y, cv=cv, scoring={'exact': 'accuracy', 'within1': within_one})
folds = pd.DataFrame({'model exact': m['test_exact'], 'baseline exact': b['test_exact'],
                      'model within-1': m['test_within1'], 'baseline within-1': b['test_within1']}, index=[f'fold {i+1}' for i in range(5)])
print(folds.round(3)); print('\nmean:', folds.mean().round(3).to_dict())

pred = cross_val_predict(model, X, y, cv=cv)
cm = confusion_matrix(y, pred, normalize='true')
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), gridspec_kw={'width_ratios': [1.2, 1]})
x_ = np.arange(5); w = 0.38
axes[0].bar(x_ - w/2, folds['baseline within-1'], w - 0.03, color=LIGHT, label='baseline (guess)')
axes[0].bar(x_ + w/2, folds['model within-1'], w - 0.03, color=BLUE, label='our model')
axes[0].set_xticks(x_, folds.index); axes[0].set_ylim(0, 1); axes[0].grid(axis='x', visible=False)
axes[0].set_ylabel('within-one-band accuracy'); axes[0].set_title('Model beats guessing in every unseen fold'); axes[0].legend(loc='lower right')
axes[1].grid(False); im = axes[1].imshow(cm, cmap=SEQ, vmin=0, vmax=1)
for i in range(6):
    for j in range(6):
        axes[1].text(j, i, f'{cm[i,j]:.0%}', ha='center', va='center', fontsize=8, color='white' if cm[i, j] > 0.5 else INK)
axes[1].set_xticks(range(6), BANDS); axes[1].set_yticks(range(6), BANDS)
axes[1].set_xlabel('predicted band'); axes[1].set_ylabel('true band'); axes[1].set_title('Most errors land in the next band')
plt.tight_layout(); plt.show()
adj = np.mean(np.abs(y - pred) <= 1)
verdict('R6 Band model', 'within-one accuracy (CV)', f"{folds['model within-1'].mean():.2f} vs {folds['baseline within-1'].mean():.2f}", 'beats baseline in all 5 folds',
        (folds['model within-1'] > folds['baseline within-1']).all())
verdict('R6 Band model', 'exact accuracy (CV)', f"{folds['model exact'].mean():.2f} vs {folds['baseline exact'].mean():.2f}", 'beats baseline in all 5 folds',
        (folds['model exact'] > folds['baseline exact']).all())

**In simple words:** on job postings it has never seen, the model is right (or one band off) about 83% of the time, and it beats guessing every single time. When it is wrong, it is usually off by only one band. **The claim holds.**

# Stage 2: Growing as a junior (which skills get a high hike?)

## Result 7: Maths/stats and storytelling matter most
**Claim:** maths/stats (**×6.2** per point) and storytelling (**×3.9**) are the strongest drivers of a high hike. Big data alone makes no difference.

**How we test it:**
1. **Box plots:** compare skill scores of high-hike vs low-hike juniors.
2. **Mann-Whitney test** for each skill on its own, with Bonferroni correction.
3. **Logistic regression** for each skill's effect *with the others held fixed* (odds ratio and 95% CI).

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(15, 3.6), sharey=True)
for ax, c in zip(axes, SKILLS):
    bp = ax.boxplot([jds.loc[jds.hike_high == 0, c], jds.loc[jds.hike_high == 1, c]], widths=0.55, patch_artist=True,
                    medianprops={'color': INK, 'lw': 2}, flierprops={'markersize': 3})
    for patch, col in zip(bp['boxes'], [BLUE, ORANGE]): patch.set_facecolor(col); patch.set_alpha(0.75)
    p = stats.mannwhitneyu(jds.loc[jds.hike_high == 1, c], jds.loc[jds.hike_high == 0, c])[1] * 5
    ax.set_xticks([1, 2], ['low\nhike', 'high\nhike']); ax.grid(axis='x', visible=False)
    ax.set_title(c.replace('_skills', '').replace('_and_', ' & ').replace('_', ' '), fontsize=10)
    ax.set_xlabel(f'p = {min(p, 1):.4f}' + ('  ✓' if p < 0.05 else '  (not sig.)'), fontsize=9)
axes[0].set_ylabel('skill rating (1–5)')
fig.suptitle('Skill ratings: low-hike (blue) vs high-hike (orange) juniors', x=0.01, ha='left', fontweight='bold')
plt.tight_layout(); plt.show()

mw = {c: min(stats.mannwhitneyu(jds.loc[jds.hike_high == 1, c], jds.loc[jds.hike_high == 0, c])[1] * 5, 1) for c in SKILLS}
logit = smf.logit('hike_high ~ ' + ' + '.join(SKILLS), data=jds).fit(disp=0)
orj = pd.DataFrame({'odds_ratio': np.exp(logit.params), 'ci_low': np.exp(logit.conf_int()[0]),
                    'ci_high': np.exp(logit.conf_int()[1]), 'p_value': logit.pvalues}).drop('Intercept')
forest(orj, 'Odds of a high hike per +1 skill point (all skills together)', 'odds ratio')
plt.show()
print(orj.round(3)); print(f'pseudo-R² = {logit.prsquared:.2f}')

verdict('R7 Junior skills', 'strongest odds ratio', f"{orj.odds_ratio.idxmax()} ×{orj.odds_ratio.max():.1f}", 'maths_stats is top', orj.odds_ratio.idxmax() == 'maths_stats_skills')
verdict('R7 Junior skills', 'storytelling odds ratio', f"×{orj.loc['dashboard_and_storytelling_skills','odds_ratio']:.1f}, p={orj.loc['dashboard_and_storytelling_skills','p_value']:.4f}", 'p < 0.05',
        orj.loc['dashboard_and_storytelling_skills', 'p_value'] < 0.05)
verdict('R7 Junior skills', 'big data alone (Mann-Whitney)', f"p = {mw['big_data_skills']:.2f}", 'p ≥ 0.05 (no effect alone)', mw['big_data_skills'] >= 0.05)

**In simple words:** the high-hike boxes sit clearly higher for maths, storytelling, coding and AI/ML, but **not** for big data. In the combined model, maths/stats has the biggest effect. **The claim holds.**

## Result 8: Hikes are very predictable
**Claim:** the model predicts high vs low hike with about **86% accuracy (AUC 0.90)**, and the simple rule *storytelling > 4.15 and maths > 3.55* works.

**How we test it:**
1. **ROC curve** from cross-validated predictions (the model never sees the person it predicts).
2. **Permutation test:** shuffle the hike labels 200 times and retrain. A real model should beat every shuffled version.
3. **Check the rule** by counting people directly.

In [ ]:
Xj, yj = jds[SKILLS], jds['hike_high']
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
cv5 = StratifiedKFold(5, shuffle=True, random_state=RNG)
proba = cross_val_predict(lr, Xj, yj, cv=cv5, method='predict_proba')[:, 1]
auc = roc_auc_score(yj, proba); acc = ((proba >= 0.5) == yj).mean()
score, perm_scores, p_perm = permutation_test_score(lr, Xj, yj, cv=cv5, n_permutations=200, scoring='roc_auc', random_state=RNG)
print(f'Cross-validated accuracy = {acc:.1%}, AUC = {auc:.3f}')
print(f'Permutation test: real AUC {score:.3f} vs shuffled max {perm_scores.max():.3f}, p = {p_perm:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
fpr, tpr, _ = roc_curve(yj, proba)
axes[0].plot(fpr, tpr, color=BLUE, lw=2.5, label=f'our model (AUC {auc:.2f})')
axes[0].plot([0, 1], [0, 1], color=LIGHT, ls='--', lw=1.5, label='guessing (AUC 0.50)')
axes[0].set_xlabel('false positive rate'); axes[0].set_ylabel('true positive rate'); axes[0].set_title('ROC curve: junior hike model'); axes[0].legend(loc='lower right')
axes[1].hist(perm_scores, bins=25, color=LIGHT, edgecolor='#fcfcfb', label='200 shuffled-label models')
axes[1].axvline(score, color=ORANGE, lw=2.5, label=f'real model {score:.2f}')
axes[1].set_xlabel('AUC'); axes[1].set_title('Real model vs shuffled-label models'); axes[1].legend(loc='upper left')
plt.tight_layout(); plt.show()

rule = (jds.dashboard_and_storytelling_skills > 4.15) & (jds.maths_stats_skills > 3.55)
low_story = jds.dashboard_and_storytelling_skills <= 4.15
print(f'Rule TRUE : {int(jds.loc[rule,"hike_high"].sum())} of {int(rule.sum())} got a high hike ({jds.loc[rule,"hike_high"].mean():.0%})')
print(f'Storytelling ≤ 4.15: {int((1-jds.loc[low_story,"hike_high"]).sum())} of {int(low_story.sum())} got a low hike ({1-jds.loc[low_story,"hike_high"].mean():.0%})')
verdict('R8 Junior model', 'cross-validated AUC', f'{auc:.2f}', '≥ 0.85', auc >= 0.85)
verdict('R8 Junior model', 'permutation test p', f'{p_perm:.4f}', 'p < 0.05', p_perm < 0.05)
verdict('R8 Junior model', 'rule precision', f"{jds.loc[rule,'hike_high'].mean():.0%} high hike", '≥ 75%', jds.loc[rule, 'hike_high'].mean() >= 0.75)

**In simple words:** the real model's AUC sits far to the right of all 200 shuffled versions, so it did not get lucky. The simple storytelling + maths rule is correct for most people it covers. **The claim holds.**

# Stage 3: Succeeding as a senior (which personality traits matter?)

## Result 9: Openness and conscientiousness define success
**Claim:** openness (**×14.7** per 10 points) and conscientiousness (**×12.0**) are the strongest traits, and neuroticism alone makes no difference.

**How we test it:** box plots, a Mann-Whitney test per trait (Bonferroni), and logistic regression odds ratios per +10 points.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(15, 3.6), sharey=True)
for ax, c in zip(axes, TRAITS):
    bp = ax.boxplot([sds.loc[sds.success_high == 0, c], sds.loc[sds.success_high == 1, c]], widths=0.55, patch_artist=True,
                    medianprops={'color': INK, 'lw': 2}, flierprops={'markersize': 3})
    for patch, col in zip(bp['boxes'], [BLUE, ORANGE]): patch.set_facecolor(col); patch.set_alpha(0.75)
    p = min(stats.mannwhitneyu(sds.loc[sds.success_high == 1, c], sds.loc[sds.success_high == 0, c])[1] * 5, 1)
    ax.set_xticks([1, 2], ['low\nsuccess', 'high\nsuccess']); ax.grid(axis='x', visible=False)
    ax.set_title(c.replace('_to_experience', '').replace('_', ' '), fontsize=10)
    ax.set_xlabel(f'p = {p:.4f}' + ('  ✓' if p < 0.05 else '  (not sig.)'), fontsize=9)
axes[0].set_ylabel('trait score')
fig.suptitle('Personality scores: low-success (blue) vs high-success (orange) seniors', x=0.01, ha='left', fontweight='bold')
plt.tight_layout(); plt.show()

mw_s = {c: min(stats.mannwhitneyu(sds.loc[sds.success_high == 1, c], sds.loc[sds.success_high == 0, c])[1] * 5, 1) for c in TRAITS}
s10 = sds.copy(); s10[TRAITS] = s10[TRAITS] / 10
logit_s = smf.logit('success_high ~ ' + ' + '.join(TRAITS), data=s10).fit(disp=0)
ors = pd.DataFrame({'odds_ratio': np.exp(logit_s.params), 'ci_low': np.exp(logit_s.conf_int()[0]),
                    'ci_high': np.exp(logit_s.conf_int()[1]), 'p_value': logit_s.pvalues}).drop('Intercept')
forest(ors, 'Odds of high success per +10 trait points (all traits together)', 'odds ratio')
plt.show()
print(ors.round(3))
top2 = ors.odds_ratio.nlargest(2).index.tolist()
verdict('R9 Senior traits', 'top-2 odds ratios', ', '.join(f'{t} ×{ors.loc[t,"odds_ratio"]:.1f}' for t in top2), 'openness & conscientiousness',
        set(top2) == {'openness_to_experience', 'conscientiousness'})
verdict('R9 Senior traits', 'both significant', f"p = {ors.loc['openness_to_experience','p_value']:.1e}, {ors.loc['conscientiousness','p_value']:.1e}", 'p < 0.05',
        (ors.loc[top2, 'p_value'] < 0.05).all())
verdict('R9 Senior traits', 'neuroticism alone (Mann-Whitney)', f"p = {mw_s['neuroticism']:.2f}", 'p ≥ 0.05 (no effect alone)', mw_s['neuroticism'] >= 0.05)

**In simple words:** the high-success boxes are far higher for openness and conscientiousness. For neuroticism the boxes overlap almost completely, so there is no difference on its own. **The claim holds.**
*Note:* in the combined model neuroticism shows OR > 1. This is a statistical side effect of holding the other traits fixed, not a real advantage, so we do not claim it.

## Result 10: Success can be predicted very accurately
**Claim:** senior success can be predicted with about **95% accuracy (AUC 0.99)**, and *openness ≤ 38.5* means **0 of 52** were successful.

**How we test it:** a cross-validated ROC curve for Random Forest, a permutation test (200 shuffles) and a direct count for the rule.

In [ ]:
Xs, ys = sds[TRAITS], sds['success_high']
rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=RNG, n_jobs=-1)
proba_s = cross_val_predict(rf, Xs, ys, cv=cv5, method='predict_proba')[:, 1]
auc_s = roc_auc_score(ys, proba_s); acc_s = ((proba_s >= 0.5) == ys).mean()
score_s, perm_s, p_perm_s = permutation_test_score(rf, Xs, ys, cv=cv5, n_permutations=200, scoring='roc_auc', random_state=RNG, n_jobs=-1)
print(f'Cross-validated accuracy = {acc_s:.1%}, AUC = {auc_s:.3f}')
print(f'Permutation test: real AUC {score_s:.3f} vs shuffled max {perm_s.max():.3f}, p = {p_perm_s:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
fpr, tpr, _ = roc_curve(ys, proba_s)
axes[0].plot(fpr, tpr, color=BLUE, lw=2.5, label=f'Random Forest (AUC {auc_s:.2f})')
axes[0].plot([0, 1], [0, 1], color=LIGHT, ls='--', lw=1.5, label='guessing (AUC 0.50)')
axes[0].set_xlabel('false positive rate'); axes[0].set_ylabel('true positive rate'); axes[0].set_title('ROC curve: senior success model'); axes[0].legend(loc='lower right')
axes[1].hist(perm_s, bins=25, color=LIGHT, edgecolor='#fcfcfb', label='200 shuffled-label models')
axes[1].axvline(score_s, color=ORANGE, lw=2.5, label=f'real model {score_s:.2f}')
axes[1].set_xlabel('AUC'); axes[1].set_title('Real model vs shuffled-label models'); axes[1].legend(loc='upper left')
plt.tight_layout(); plt.show()

low_open = sds.openness_to_experience <= 38.5
print(f'Openness ≤ 38.5: {int(sds.loc[low_open,"success_high"].sum())} of {int(low_open.sum())} were successful')
verdict('R10 Senior model', 'cross-validated AUC', f'{auc_s:.2f}', '≥ 0.95', auc_s >= 0.95)
verdict('R10 Senior model', 'permutation test p', f'{p_perm_s:.4f}', 'p < 0.05', p_perm_s < 0.05)
verdict('R10 Senior model', 'low-openness rule', f"{int(sds.loc[low_open,'success_high'].sum())} of {int(low_open.sum())} successful", '0 successful', sds.loc[low_open, 'success_high'].sum() == 0)

**In simple words:** the ROC curve hugs the top-left corner, which means near-perfect separation, and the real model is far better than every shuffled one. Nobody with low openness was successful. **The claim holds.**

## Result 11: Four personality types with very different success
**Claim:** grouping seniors by personality **without** looking at success still produces groups with very different success rates (89% vs 0%).

**How we test it:**
1. **Chi-square test:** is success rate linked to the group?
2. **Stability check:** rerun the clustering with 20 different random starts and measure how similar the groups are (Adjusted Rand Index; 1 = identical).

In [ ]:
Z = StandardScaler().fit_transform(Xs)
km = KMeans(4, n_init=20, random_state=RNG).fit(Z); sds['persona'] = km.labels_
prof = pd.DataFrame(Z, columns=TRAITS).groupby(sds['persona']).mean()
def name(r):
    if r['conscientiousness'] > 0.4 and r['openness_to_experience'] > 0.4: return 'Conscientious & open'
    if r.mean() < -0.6: return 'Low on all traits'
    return f"High {r.idxmax().split('_')[0]}, low {r.idxmin().split('_')[0]}"
names = prof.apply(name, axis=1)
tab = pd.DataFrame({'type': names, 'people': sds.persona.value_counts().sort_index(),
                    'success_%': sds.groupby('persona').success_high.mean().mul(100).round(0)}).sort_values('success_%', ascending=False)
print(tab)
chi2, p_chi, _, _ = stats.chi2_contingency(pd.crosstab(sds.persona, sds.success_high))
ari = [adjusted_rand_score(km.labels_, KMeans(4, n_init=20, random_state=s).fit_predict(Z)) for s in range(20)]
print(f'Chi-square: p = {p_chi:.1e} | stability (ARI over 20 seeds): mean {np.mean(ari):.2f}, min {np.min(ari):.2f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8), gridspec_kw={'width_ratios': [2.2, 1]})
P = prof.loc[tab.index]; axes[0].grid(False)
im = axes[0].imshow(P.values, cmap=DIV, vmin=-1.5, vmax=1.5, aspect='auto')
for i in range(P.shape[0]):
    for j in range(5): axes[0].text(j, i, f'{P.values[i, j]:+.1f}', ha='center', va='center', fontsize=9, color=INK)
axes[0].set_xticks(range(5), ['neurotic.', 'extravers.', 'openness', 'agreeable.', 'conscient.'])
axes[0].set_yticks(range(len(tab)), [f"{t} (n={n})" for t, n in zip(tab['type'], tab['people'])])
axes[0].set_title('Trait profile of each type (orange = above average)'); plt.colorbar(im, ax=axes[0], fraction=0.03)
s = tab['success_%'][::-1]
axes[1].barh(range(len(s)), s.values, color=BLUE, height=0.6); axes[1].set_yticks(range(len(s)), [''] * len(s))
for i, v_ in enumerate(s.values): axes[1].text(v_, i, f' {v_:.0f}%', va='center', fontsize=9, color=MUTED)
axes[1].set_xlim(0, 110); axes[1].grid(axis='y', visible=False); axes[1].set_xlabel('% high success'); axes[1].set_title('Success rate')
plt.tight_layout(); plt.show()
verdict('R11 Personas', 'chi-square type vs success', f'p = {p_chi:.1e}', 'p < 0.05', p_chi < 0.05)
verdict('R11 Personas', 'best vs worst success rate', f"{tab['success_%'].max():.0f}% vs {tab['success_%'].min():.0f}%", 'gap ≥ 50 points', tab['success_%'].max() - tab['success_%'].min() >= 50)
verdict('R11 Personas', 'cluster stability (ARI)', f'{np.mean(ari):.2f}', '≥ 0.8', np.mean(ari) >= 0.8)

**In simple words:** the groups were formed without seeing success, yet one type succeeds 89% of the time and two succeed 0% of the time. The same groups come back every time we rerun the clustering. **The claim holds.**

# Scorecard: every check in one table

In [ ]:
card = pd.DataFrame(CHECKS)
print(f"{(card.verdict == '✅ PASS').sum()} of {len(card)} checks passed")
card

In [ ]:
res = card.groupby('result')['verdict'].apply(lambda v: (v == '✅ PASS').mean() * 100)
order = sorted(res.index, key=lambda r: int(re.search(r'R(\d+)', r).group(1)))
res = res.loc[order]
fig, ax = plt.subplots(figsize=(8, 4.5)); y = np.arange(len(res))[::-1]
ax.barh(y, res.values, color=[AQUA if v == 100 else ORANGE for v in res.values], height=0.6)
for yi, v_ in zip(y, res.values): ax.text(v_, yi, f' {v_:.0f}%', va='center', fontsize=9, color=MUTED)
ax.set_yticks(y, res.index); ax.set_xlim(0, 112); ax.grid(axis='y', visible=False)
ax.set_xlabel('% of checks passed'); ax.set_title('Verification scorecard: share of checks passed per result')
plt.show()

## Conclusion
Every result in `results_and_conclusions.md` was tested in at least two independent ways: a statistical test **and** a robustness check (bootstrap, cross-validation, permutation test or stability check).
Results that pass all their checks are **not due to chance, not over-fitted, and stable when the data is re-sampled**.

**Reminder about limits:** these tests prove the patterns are *real in this data*. They show **association, not cause**, and the JDS and SDS samples come from one company.